# Markov Chains and Mixing Times

| Difficulty | ██████░░░░ 6/10 |
| :--- | :--- |
| Prerequisites | 09.02 |
| Time | ~75 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/09_Advanced_Probability/03_markov_chains_and_mixing.ipynb)

---

## 🎯 Learning Objective

Study transition matrices, stationary distributions, and mixing times, and understand ergodicity.

---

## 📐 Theory

`09.02` studied Brownian motion, a *continuous*-state, *continuous*-time Markov process. This
notebook studies the discrete-state, discrete-time case — simpler to compute with exactly, and
the direct engine behind MCMC (`03.09`), PageRank, and autoregressive text generation.

### Markov chains and transition matrices

A **Markov chain** is a sequence of random variables $X_0, X_1, X_2, \ldots$ taking values in a
finite state space $\{1,\ldots,n\}$, satisfying the **Markov property**: the future depends on
the past only through the present,

$$P(X_{t+1} = j \mid X_t = i, X_{t-1}, \ldots, X_0) = P(X_{t+1} = j \mid X_t = i) =: P_{ij}.$$

Collecting these one-step probabilities into an $n\times n$ **transition matrix** $P$ (row $i$ =
the distribution of where you go next, given you're at state $i$; every row sums to $1$ — a
**row-stochastic** matrix) turns the entire dynamics into linear algebra. If $\mu_t$ is the
(row-vector) distribution of $X_t$ over states, then $\mu_{t+1} = \mu_t P$ — one matrix
multiplication advances the whole distribution by one step, and $\mu_t = \mu_0 P^t$ jumps
straight to $t$ steps ahead. This is the discrete-time, discrete-state analogue of the SDE
machinery from `09.02`; a Wiener process is, in a sense, what you get taking the state space
continuous and the time step to zero.

### Stationary distributions

A distribution $\pi$ (row vector, entries summing to $1$) is **stationary** for $P$ if
$\pi P = \pi$ — once you reach $\pi$, every subsequent step leaves the distribution unchanged
(individual states still hop around; the *population-level* distribution over states is fixed).
Transposing both sides of $\pi P = \pi$ and using $(\pi P)^\top = P^\top \pi^\top$ (the
transpose-of-a-product rule) gives $P^\top \pi^\top = \pi^\top$, i.e. $\pi^\top$ is a genuine
eigenvector of $P^\top$ with eigenvalue $1$, so computing $\pi$ reduces to eigendecomposition
(`01.06`) — one of two methods this notebook implements and cross-checks against each other. The
**Perron-Frobenius theorem** guarantees that for any row-stochastic $P$, an eigenvalue-$1$
eigenvector with non-negative entries always exists, and if the chain is **irreducible** (every
state reachable from every other) it is unique up to scaling — so normalizing it to sum to $1$
pins down $\pi$.

### Ergodicity and convergence to stationarity

A chain is **ergodic** if it is irreducible and **aperiodic** (it doesn't cycle through states on
a fixed rhythm — having any state with a self-loop, i.e. $P_{ii}>0$, already rules out
periodicity). The **ergodic theorem for Markov chains** then guarantees two independently useful
facts, both verified numerically below: (1) *regardless of the starting distribution* $\mu_0$,
the chain's distribution $\mu_0 P^t \to \pi$ as $t\to\infty$ — so $\pi$ can be estimated by
simply running the chain for a long time and recording how often each state is visited, no
eigendecomposition needed; and (2) time averages converge to the stationary expectation almost
surely (`09.01`'s language) — the Markov-chain analogue of the Law of Large Numbers, and exactly
what justifies treating MCMC samples (`03.09`) as representative draws from the target
distribution once the chain has run long enough.

### Mixing time: how long is "long enough"?

**Total variation (TV) distance** between two distributions on the same finite state space is
$\|\mu-\nu\|_{TV} = \frac12\sum_i |\mu_i - \nu_i|$ — half the $L^1$ distance between them, always
between $0$ (identical) and $1$ (disjoint support). The **mixing time** $t_{\text{mix}}(\epsilon)$
is the first time $t$ such that $\|\mu_0 P^t - \pi\|_{TV} < \epsilon$ for every possible starting
distribution $\mu_0$ — the point at which the chain has (nearly) forgotten where it started. For
an ergodic chain, $\|\mu_0 P^t - \pi\|_{TV}$ decays *exponentially* in $t$, at a rate governed by
the transition matrix's **second-largest eigenvalue modulus (SLEM)**, $|\lambda_2|$ (the
largest-magnitude eigenvalue other than the guaranteed $\lambda_1=1$): roughly,
$\|\mu_0 P^t - \pi\|_{TV} \propto |\lambda_2|^t$. A SLEM close to $1$ means slow mixing (the chain
has some near-stationary direction it escapes only slowly, often because of a "bottleneck"
separating two weakly-connected clusters of states); a small SLEM means fast mixing. This
spectral-gap-controls-convergence-speed pattern is a discrete, finite-state preview of the same
idea behind how MCMC samplers (`03.09`) are diagnosed for "have I run this long enough" in
practice.

### PageRank: a stationary distribution, applied to the entire web

**PageRank** models a "random surfer" clicking links: from any web page, follow an outgoing link
uniformly at random (with a small probability $1-d$ of "teleporting" to a uniformly random page
instead, guaranteeing irreducibility even if the link graph itself isn't strongly connected).
This defines a genuine Markov chain — the **Google matrix**
$G = d\,P + (1-d)\,\tfrac{1}{n}\mathbf{1}\mathbf{1}^\top$, where $P$ is the link-following
transition matrix and $d\approx 0.85$ — and a page's **PageRank score** is defined to be exactly
its stationary probability $\pi_i$: the long-run fraction of clicks that land on page $i$. This
notebook implements PageRank as nothing more than a literal instance of the stationary
distribution computed above, on a graph built with the same `networkx` tools as `07.02`.

---

In [ ]:
# Setup
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting total variation distance to stationarity over time, for a well-connected chain versus
one with a weak bottleneck between two clusters of states, makes "mixing time" and "spectral
gap" visible as the same phenomenon: the bottlenecked chain's TV distance decays exponentially,
but at a visibly slower rate.

In [ ]:
# Two Markov chains, same recipe for computing stationarity, very different mixing speed:
# a well-connected 5-state chain vs. a 6-state chain with two clusters joined by a weak bridge.
rng = np.random.default_rng(0)

def stationary_via_eigenvector(P, eigval_tol=1e-6):
    # eig (not eigh) is the correct choice here: P is row-stochastic but generally NOT
    # symmetric, so eigh's symmetry assumption would silently give a wrong answer.
    eigvals, eigvecs = np.linalg.eig(P.T)
    idx = np.argmin(np.abs(eigvals - 1))    # the eigenvalue guaranteed to be exactly 1 (Perron-Frobenius)

    # Guard 1: confirm the CLOSEST eigenvalue is actually close to 1, not just the closest
    # of a bunch of eigenvalues all far from it. Perron-Frobenius guarantees an eigenvalue
    # of exactly 1 exists for a stochastic matrix, so a large gap here would mean something
    # is wrong with P itself (e.g. rows don't sum to 1), not with the eigenvector selection.
    if np.abs(eigvals[idx] - 1) > eigval_tol:
        raise ValueError(
            f"No eigenvalue of P^T is within {eigval_tol} of 1 (closest: {eigvals[idx]:.6f}). "
            f"P may not be a valid row-stochastic matrix (Perron-Frobenius guarantees an "
            f"eigenvalue of exactly 1 for one that is)."
        )

    pi = np.real(eigvecs[:, idx])

    # Guard 2: for an irreducible chain, Perron-Frobenius guarantees the eigenvalue-1
    # eigenvector can be chosen with ALL-NONNEGATIVE entries -- but np.linalg.eig returns
    # an arbitrarily-scaled (and arbitrarily-SIGNED) eigenvector, so pi could come back
    # all-negative (self-corrects when we divide by a negative sum) or, if numerical noise
    # ever picked a slightly-wrong eigenvector, genuinely mixed-sign (does NOT self-correct:
    # pi.sum() could then be small or even ~0, and dividing by it would blow up nonsensically
    # with no error raised). np.abs() is safe specifically BECAUSE Perron-Frobenius already
    # guarantees the correct eigenvector is one-signed -- we are only ever flipping every
    # entry's sign together (harmless) or, if the guard above already passed, confirming
    # consistency rather than papering over a genuine error.
    pi = np.abs(pi)
    pi_sum = pi.sum()
    if pi_sum < eigval_tol:
        raise ValueError(
            f"Eigenvector sum ({pi_sum:.2e}) is too close to zero to normalize safely -- "
            f"the selected eigenvector does not look like a valid Perron-Frobenius vector."
        )
    return pi / pi_sum

def tv_distance(p, q):
    return 0.5 * np.sum(np.abs(p - q))

# Well-connected chain: random transitions, no obvious bottleneck
n_fast = 5
raw = rng.uniform(0.1, 1.0, size=(n_fast, n_fast))
P_fast = raw / raw.sum(axis=1, keepdims=True)
pi_fast = stationary_via_eigenvector(P_fast)

# --- Regression test: stationary_via_eigenvector must reject a matrix that ISN'T actually
# row-stochastic (rows don't sum to 1), rather than silently returning nonsense. ---
P_not_stochastic = np.array([[0.5, 0.3], [0.2, 0.5]])   # rows sum to 0.8 and 0.7, not 1
try:
    stationary_via_eigenvector(P_not_stochastic)
    print("FAILED to reject a non-stochastic matrix (this should not happen)")
except ValueError as e:
    print(f"Regression test: non-stochastic matrix correctly rejected -- {e}")

# Bottlenecked chain: two 3-state clusters, strong within-cluster transitions, weak across
n_slow = 6
P_slow = np.zeros((n_slow, n_slow))
within, cross = 0.30, 0.02
for i in range(n_slow):
    cluster_i = 0 if i < 3 else 1
    for j in range(n_slow):
        cluster_j = 0 if j < 3 else 1
        if i == j:
            P_slow[i, j] = 0.5             # "lazy" self-loop, also guarantees aperiodicity
        elif cluster_j == cluster_i:
            P_slow[i, j] = within
        else:
            P_slow[i, j] = cross            # rare cross-cluster jump = the bottleneck
    P_slow[i] /= P_slow[i].sum()
pi_slow = stationary_via_eigenvector(P_slow)

def tv_curve(P, pi, n_iters, start_state=0):
    dist = np.zeros(len(pi)); dist[start_state] = 1.0
    curve = []
    for _ in range(n_iters):
        curve.append(tv_distance(dist, pi))
        dist = dist @ P
    return np.array(curve)

curve_fast = tv_curve(P_fast, pi_fast, 30)
curve_slow = tv_curve(P_slow, pi_slow, 60)

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.semilogy(curve_fast, "o-", color="#4C72B0", markersize=4, label="well-connected 5-state chain")
ax.semilogy(curve_slow, "s-", color="#C44E52", markersize=4, label="bottlenecked 6-state chain (2 clusters)")
ax.axhline(0.25, color="gray", ls="--", lw=1, label="eps=0.25 threshold")
ax.set_xlabel("t (steps)"); ax.set_ylabel("TV distance to stationary pi  (log scale)")
ax.set_title("Mixing time: TV distance decays EXPONENTIALLY,\nbut a bottleneck makes the rate much slower")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

eps = 0.25
t_mix_fast = next(t for t, v in enumerate(curve_fast) if v < eps)
t_mix_slow = next(t for t, v in enumerate(curve_slow) if v < eps)
print(f"t_mix(0.25) for the well-connected chain: {t_mix_fast} steps")
print(f"t_mix(0.25) for the bottlenecked chain:   {t_mix_slow} steps")
print(f"The bottlenecked chain takes {t_mix_slow/t_mix_fast:.1f}x longer to mix -- a direct,")
print("visible consequence of the weak cross-cluster transition probability, quantified")
print("rigorously below via the transition matrix's second-largest eigenvalue.")

## 🐍 Implementation from Scratch

We compute a Markov chain's stationary distribution two independent ways (eigenvector of $P^\top$
vs. long-run empirical simulation) and confirm they agree, then verify that the total variation
decay rate matches the transition matrix's second-largest eigenvalue exactly — not just
approximately — before treating that as a general "mixing rate" fact.

In [ ]:
# --- Stationary distribution, computed TWO independent ways ---
def stationary_via_eigenvector(P, eigval_tol=1e-6):
    # pi P = pi means pi^T is an eigenvector of P^T with eigenvalue 1 (Perron-Frobenius
    # guarantees this eigenvalue exists and, for an irreducible chain, is unique up to scale).
    # eig, not eigh: P is generally non-symmetric, so eigh's symmetry assumption would be wrong.
    eigvals, eigvecs = np.linalg.eig(P.T)
    idx = np.argmin(np.abs(eigvals - 1))

    # Same two guards as the Visual Intuition section's copy of this function: (1) confirm
    # the selected eigenvalue really is close to 1 rather than just the closest available one,
    # and (2) use abs() (justified by Perron-Frobenius guaranteeing a one-signed eigenvector
    # for an irreducible chain) plus a zero-sum guard before normalizing, so a numerically
    # wrong eigenvector selection raises a clear error instead of silently blowing up.
    if np.abs(eigvals[idx] - 1) > eigval_tol:
        raise ValueError(
            f"No eigenvalue of P^T is within {eigval_tol} of 1 (closest: {eigvals[idx]:.6f}) -- "
            f"P may not be a valid row-stochastic matrix."
        )
    pi = np.abs(np.real(eigvecs[:, idx]))
    pi_sum = pi.sum()
    if pi_sum < eigval_tol:
        raise ValueError(f"Eigenvector sum ({pi_sum:.2e}) too close to zero to normalize safely.")
    return pi / pi_sum

def stationary_via_simulation(P, n_steps, rng, burn_in=1000):
    # The ergodic theorem's OTHER guarantee: long-run visit frequencies converge to pi,
    # regardless of start state -- no eigendecomposition needed, just running the chain.
    n = len(P)
    state = 0
    counts = np.zeros(n)
    for t in range(n_steps + burn_in):
        state = rng.choice(n, p=P[state])
        if t >= burn_in:            # discard the transient before the chain has "forgotten" state 0
            counts[state] += 1
    return counts / n_steps

rng = np.random.default_rng(0)
n = 5
raw = rng.uniform(0.1, 1.0, size=(n, n))
P = raw / raw.sum(axis=1, keepdims=True)

pi_eig = stationary_via_eigenvector(P)
pi_sim = stationary_via_simulation(P, n_steps=300_000, rng=np.random.default_rng(1))
print("Stationary distribution via eigenvector: ", np.round(pi_eig, 4))
print("Stationary distribution via simulation:  ", np.round(pi_sim, 4))
print(f"Max absolute difference: {np.max(np.abs(pi_eig - pi_sim)):.5f}  (two independent methods, close agreement)")
print(f"Sanity check pi @ P == pi (eigenvector method): {np.allclose(pi_eig @ P, pi_eig, atol=1e-8)}")
print(f"Sanity check P.T @ pi.T == pi.T (the actual eigenvector equation for P^T used above): "
      f"{np.allclose(P.T @ pi_eig, pi_eig, atol=1e-8)}")

# --- Mixing time and the spectral gap: does the TV decay rate ACTUALLY equal the SLEM, or is
# that just a plausible-sounding rule of thumb? Fit the decay rate directly and compare. ---
n_slow = 6
P_slow = np.zeros((n_slow, n_slow))
within, cross = 0.30, 0.02
for i in range(n_slow):
    cluster_i = 0 if i < 3 else 1
    for j in range(n_slow):
        cluster_j = 0 if j < 3 else 1
        if i == j:
            P_slow[i, j] = 0.5
        elif cluster_j == cluster_i:
            P_slow[i, j] = within
        else:
            P_slow[i, j] = cross
    P_slow[i] /= P_slow[i].sum()
pi_slow = stationary_via_eigenvector(P_slow)

def tv_distance(p, q):
    return 0.5 * np.sum(np.abs(p - q))

dist = np.zeros(n_slow); dist[0] = 1.0
tv_curve = []
for _ in range(60):
    tv_curve.append(tv_distance(dist, pi_slow))
    dist = dist @ P_slow
tv_curve = np.array(tv_curve)

eigvals_slow = np.linalg.eigvals(P_slow)
eigvals_sorted = sorted(eigvals_slow, key=lambda x: -abs(x))
slem = abs(eigvals_sorted[1])   # second-largest eigenvalue MODULUS (the largest one besides 1)

# Fit log(TV(t)) ~ t*log(rate) + const over an ASYMPTOTIC window (skip the early transient,
# where sub-dominant eigenvalues besides the SLEM still contribute) -- an independent estimate
# of the true decay rate, to check against the SLEM rather than assume the match.
fit_window = np.arange(15, 45)
slope, intercept = np.polyfit(fit_window, np.log(tv_curve[fit_window]), 1)
fitted_rate = np.exp(slope)

print(f"\nSecond-largest eigenvalue modulus (SLEM) of P_slow: {slem:.6f}")
print(f"TV-distance decay rate, fit independently from the simulated curve: {fitted_rate:.6f}")
print(f"Match: {np.isclose(fitted_rate, slem, rtol=1e-3)}  "
      f"(confirms mixing speed IS governed by the spectral gap 1-|lambda_2|, not a coincidence)")

## 🤖 Why This Matters for AI

MCMC convergence diagnostics, PageRank, and autoregressive text generation itself is a (very
high-order) Markov chain. PageRank is the cleanest possible example: a page's importance score
*is*, by definition, its stationary probability under a "random surfer" Markov chain over the
web graph (`07.02`'s adjacency-matrix language, applied to hyperlinks). We implement it below as
nothing more than an instance of `stationary_via_eigenvector`, and cross-check it against
`networkx.pagerank`'s independent implementation and against power iteration (repeatedly
multiplying by the transition matrix) — the same three-way check used above for the toy chain,
now applied to a genuine graph problem.

In [ ]:
# PageRank = the stationary distribution of a "random surfer" Markov chain over a link graph.
# Not an analogy -- this cell computes it as a LITERAL instance of stationary_via_eigenvector.
rng = np.random.default_rng(3)
G = nx.gnp_random_graph(8, 0.3, directed=True, seed=3)
G.remove_edges_from(nx.selfloop_edges(G))   # a page linking to itself isn't a meaningful "vote"

n_pages = G.number_of_nodes()
A = nx.to_numpy_array(G, nodelist=sorted(G.nodes()))
out_degree = A.sum(axis=1)

damping = 0.85   # standard PageRank damping factor
P_link = np.zeros((n_pages, n_pages))
for i in range(n_pages):
    if out_degree[i] == 0:
        P_link[i, :] = 1.0 / n_pages     # dangling page (no outlinks): teleport uniformly
    else:
        P_link[i, :] = A[i, :] / out_degree[i]   # follow an outgoing link uniformly at random

# The Google matrix: with probability `damping` follow a real link, else teleport uniformly --
# the teleport term is what GUARANTEES irreducibility (every page reachable from every other),
# which is exactly the condition Perron-Frobenius needs for a unique stationary distribution.
google_matrix = damping * P_link + (1 - damping) * np.ones((n_pages, n_pages)) / n_pages
print(f"Google matrix row sums (must all be 1): {np.round(google_matrix.sum(axis=1), 6)}")

pagerank_eig = stationary_via_eigenvector(google_matrix)

# Independent check #1: networkx's OWN pagerank implementation
pagerank_nx = nx.pagerank(G, alpha=damping)
pagerank_nx_arr = np.array([pagerank_nx[i] for i in sorted(G.nodes())])

# Independent check #2: power iteration (repeated multiplication by the Google matrix) --
# the actual algorithm real search engines used historically, requiring no eigendecomposition
v = np.ones(n_pages) / n_pages
for _ in range(300):
    v = v @ google_matrix
pagerank_power_iter = v

print(f"\n{'page':>5} | {'via eigenvector':>16} | {'via networkx':>14} | {'via power iteration':>20}")
for i in range(n_pages):
    print(f"{i:5d} | {pagerank_eig[i]:16.4f} | {pagerank_nx_arr[i]:14.4f} | {pagerank_power_iter[i]:20.4f}")

print(f"\nMax diff (eigenvector vs networkx):      {np.max(np.abs(pagerank_eig - pagerank_nx_arr)):.2e}")
print(f"Max diff (eigenvector vs power iteration): {np.max(np.abs(pagerank_eig - pagerank_power_iter)):.2e}")

top_page = np.argmax(pagerank_eig)
in_degrees = A.sum(axis=0)
print(f"\nHighest-ranked page: {top_page}, PageRank={pagerank_eig[top_page]:.4f}, "
      f"in-degree={int(in_degrees[top_page])} (max in-degree across all pages: {int(in_degrees.max())})")
print("PageRank score tracks being LINKED TO (in-degree), not how many links a page sends out --")
print("the random surfer's stationary probability accumulates wherever incoming paths converge.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. For the 2-state chain $P = \begin{pmatrix}0.9 & 0.1\\ 0.4 & 0.6\end{pmatrix}$, solve
   $\pi P = \pi$ by hand (using $\pi_1+\pi_2=1$) to find the stationary distribution. Check
   that both entries are non-negative and sum to $1$ before verifying with
   `stationary_via_eigenvector`.

<details>
<summary>💡 Solution</summary>

$\pi P=\pi$ gives $0.9\pi_1+0.4\pi_2=\pi_1$, i.e. $0.4\pi_2=0.1\pi_1$, i.e. $\pi_1=4\pi_2$.
With $\pi_1+\pi_2=1$: $5\pi_2=1\Rightarrow\pi_2=0.2,\ \pi_1=0.8$. Both entries are
non-negative and sum to exactly $1$, as required of any valid distribution.
`stationary_via_eigenvector(P)` returns $(0.8, 0.2)$, matching exactly.

</details>

### 💭 UNDERSTAND
2. `stationary_via_eigenvector` uses `np.linalg.eig`, not `np.linalg.eigh`, and the code
   comments explain why (P is generally non-symmetric). Suppose someone "simplifies" the
   function by swapping in `eigh` because it's faster and the matrix "looks" close to symmetric
   in a specific example. Using the Theory section's own claims about what `eigh` assumes,
   explain what could go wrong, even on an example where $P$ happens to look nearly symmetric.

<details>
<summary>💡 Solution</summary>

`eigh` assumes its input is *exactly* symmetric (it deliberately only reads one triangular half
and mirrors it, ignoring the other half entirely) — so if $P$ is even slightly asymmetric,
`eigh` doesn't raise an error or approximate anything; it silently computes the eigendecomposition
of a *different* matrix (the one obtained by mirroring one triangle over the other), which has
no particular relationship to $P$'s true stationary distribution. "Looks nearly symmetric" is not
a safety net here — `eigh` doesn't degrade gracefully as asymmetry grows, it's simply wrong for
any matrix that isn't exactly symmetric, silently and without warning, which is exactly why the
code deliberately chose `eig` despite it being slower.

</details>

### ✏️ DERIVE
3. The Theory section states that TV distance decays like $\|\mu_0 P^t-\pi\|_{TV} \propto
   |\lambda_2|^t$. Starting from the eigendecomposition $P^\top v_i = \lambda_i v_i$ (so $\mu_0
   P^t = \pi + \sum_{i\ge2} c_i \lambda_i^t v_i^\top$ for some coefficients $c_i$ depending on
   $\mu_0$, once $\mu_0$ is expressed in the eigenvector basis), sketch why the $\lambda_2$ term
   eventually dominates every other non-stationary term as $t\to\infty$, making the SLEM the
   right single number to characterize asymptotic mixing speed.

<details>
<summary>💡 Solution outline</summary>

Writing $\mu_0-\pi$ in the eigenbasis of $P^\top$ (excluding the $\lambda_1=1$ direction, since
that component is exactly $\pi$ itself and doesn't decay) gives
$\mu_0 P^t - \pi = \sum_{i\ge2} c_i \lambda_i^t v_i^\top$. Since $|\lambda_2|\ge|\lambda_3|\ge
\cdots$ by definition (the eigenvalues are sorted by modulus), every term with $i>2$ shrinks
*strictly faster* than the $i=2$ term as $t$ grows (a smaller base raised to a growing power
falls behind a larger base exponentially in the ratio $t$) — so for large enough $t$, the sum is
dominated by its $i=2$ term alone, giving $\|\mu_0P^t-\pi\|\approx |c_2|\cdot|\lambda_2|^t$,
exactly the claimed proportionality. This is precisely why the Implementation cell's curve-fitting
exercise deliberately skips the early iterations (`fit_window = np.arange(15, 45)`) before fitting
the decay rate — at small $t$, the sub-dominant eigenvalues haven't yet become negligible, so the
pure $|\lambda_2|^t$ approximation isn't accurate yet.

</details>

### 🐍 IMPLEMENT
4. In the mixing-time experiment, increase `cross` from `0.02` to `0.10` (a less severe
   bottleneck) and recompute the SLEM and the fitted TV-decay rate. Confirm they still match
   each other, and confirm $t_{\text{mix}}(0.25)$ is now smaller than the original bottlenecked
   chain's.

<details>
<summary>✅ How to know you're right</summary>

At `cross=0.02` (the original), SLEM and the fitted rate both come out to $\approx0.8966$, with
$t_{\text{mix}}(0.25)=7$. At `cross=0.10`, both SLEM and the fitted rate should drop to
$\approx0.5714$ (still matching each other exactly, confirming the spectral-gap relationship
holds at this new parameter too, not just the original one), and $t_{\text{mix}}(0.25)$ should
drop to $2$ — a much faster-mixing chain, as expected from weakening the bottleneck. If your
SLEM and fitted rate *don't* match at the new `cross` value, that's a sign of a bug in how you
rebuilt the chain (e.g. forgetting to renormalize rows after changing `cross`), not evidence
against the underlying theory.

</details>

### 🤖 APPLY
5. Autoregressive language models generate token $x_{t+1}$ from a distribution conditioned on
   the *entire* history $x_1,\ldots,x_t$, seemingly violating the Markov property (which only
   allows dependence on $x_t$ alone). Explain why treating the full history "so far" as a single
   Markov *state* $s_t=(x_1,\ldots,x_t)$ restores the Markov property trivially, and why this
   trivial fix doesn't help in practice — connect your answer to why real language models use a
   fixed-size hidden state or an attention window instead of a literally growing state space.

<details>
<summary>✅ What you should observe</summary>

Defining $s_t=(x_1,\ldots,x_t)$ as "the state" makes the Markov property trivially true by
construction: $s_{t+1}=(s_t, x_{t+1})$ depends only on $s_t$ (and the new token), so nothing
about the *sequence itself* violates the property — it was never really violated, only
*expressed* in terms of a growing observable ($x_t$ alone) rather than the true Markov state
($s_t$). This "fix" is practically useless because the state space $\{s_t\}$ grows without
bound as $t$ increases (there are exponentially many possible histories of length $t$), so the
transition-matrix machinery this notebook relies on (a fixed, finite $n\times n$ matrix) simply
doesn't apply — you cannot even write down $P$ for a state space this large, let alone
eigendecompose it. Real language models resolve this not by literally tracking the growing
history but by *compressing* it into a fixed-size representation (a hidden state, or attention
over a bounded context window) that approximates the sufficient statistics of the true history
well enough for the task, trading exact Markov-ness in a huge state space for an approximate,
learned Markov-like structure in a small, fixed one.

</details>

### 🚀 CHALLENGE
6. In the PageRank graph, node `4` has zero out-degree (a dangling page). Remove it entirely and
   recompute PageRank for the remaining 7 pages. Does removing a zero-out-degree node change the
   *relative order* of the remaining pages' scores?

<details>
<summary>🔍 What a strong answer covers</summary>

Before removal, the 8-page ranking (highest to lowest) is
page $1 > 2\!=\!3 > 6 > 0\!=\!5 > 4 > 7$ (pages 2/3 exactly tied, pages 0/5 exactly tied — a
symmetry of this specific random graph). After removing dangling page 4, the remaining 7 pages'
ranking is $1 > 2\!=\!3 > 6 > 0\!=\!5 > 7$ — *identical* relative order among all 7 surviving
pages, just with page 4 itself gone from the list; every surviving page's score changes
slightly (e.g. page 1 rises from $0.2033$ to $0.2207$, since a dangling page's uniform-teleport
contribution to everyone is redistributed slightly differently once it's removed), but no two
pages swap relative positions. A strong answer explains why this isn't a coincidence for this
particular graph: a dangling page contributes identically (uniformly) to every other page's
score via the teleport mechanism regardless of the specific graph structure, so removing it
shifts every remaining page's absolute score by a similar-in-kind adjustment rather than
favoring some pages over others — though a rigorous answer should note this is an empirical
observation on one graph, not a claim that removing *any* node (especially a non-dangling,
well-linked one) would similarly preserve relative order in general.

</details>

---

## 📚 Further Reading
- Levin & Peres, *Markov Chains and Mixing Times*, Ch. 1, 4, 12 (the standard modern reference)
- Page, Brin, Motwani & Winograd, ["The PageRank Citation Ranking"](http://ilpubs.stanford.edu:8090/422/) (1999)
- Grinstead & Snell, *Introduction to Probability*, Ch. 11 (Markov Chains) — free online

---

*Next notebook: [Concentration Inequalities](04_concentration_inequalities.ipynb)*